# C · Message drafts (live session)

Staff pick an allowed offer for a member and get a ready-to-edit message for one channel (W22). Nothing is sent.

Two paths, same evidence: a **deterministic template** (always works, uses only facts in the evidence) and an **LLM
rewrite** under a grounded prompt. LLM output is rejected, and the template used instead, if it mentions churn, risk or
scores, contains a placeholder, uses a number not in the evidence, or runs over the SMS limit.

Run the template cells first. Section 3 makes one LLM call: run it once and read the output for tone and invented claims.

In [1]:
from sqlmodel import Session

from anchorai.assistant import drafts
from anchorai.assistant.drafts import DraftIn
from anchorai.db.session import get_engine

session = Session(get_engine())

def show(member_id, offer_id, channel, use_llm=False):
    d = drafts.make_draft(session, DraftIn(member_id=member_id, offer_id=offer_id, channel=channel), use_llm=use_llm)
    print(f"--- {member_id} / {offer_id} / {channel} [{d.source}]")
    if d.subject:
        print("Subject:", d.subject)
    print(d.text)
    print("Evidence:", *d.evidence, sep="\n  - ")
    print()

## 1. Template drafts

What to look for: friendly, short, no prices or dates that the catalogue doesn't state, and nothing about risk.

In [2]:
show("M0042", "free_pt_session", "call")
show("M0042", "checkin_message", "sms")
show("M0155", "discount_1m", "email")

--- M0042 / free_pt_session / call [template]
Hi, this is from your gym. Do you have a minute?
We hope you're enjoying your first months with us.
We'd like to offer you: Free personal training session.
Would you like to take this up?
Thanks for your time, and see you at the gym.
Evidence:
  - Offer: Free personal training session
  - Who it is for: Members without PT in the last 90 days
  - Channel: call
  - The member has been with the gym for 1 month

--- M0042 / checkin_message / sms [template]
Hi! We hope you're enjoying your first months with us. We wanted to check in and see how your training is going. If there's anything we can help with, just reply here.
Evidence:
  - Offer: Personal check-in message
  - Who it is for: Any active member
  - Channel: sms
  - The member has been with the gym for 1 month

--- M0155 / discount_1m / email [template]
Subject: Something for you: One month at 20% off
Hi,

Welcome to the gym. We know a familiar instructor has moved on, and we'd like to 

## 2. A blocked offer is refused

Guardrails apply: drafting a message for an offer the rule base blocks raises `OfferBlocked`. An offer on a channel the catalogue doesn't list raises `OfferUnavailable`.


In [3]:
from anchorai.reasoning.service import all_facts, trace

blocked = next((m, o) for m in all_facts() for o in trace(session, m).blocked if o != "all")
try:
    show(*blocked, "email")
except drafts.OfferBlocked as e:
    print("Refused:", e)

# Each offer goes out only on its catalogue channels; PT is by call or in person.
try:
    show("M0042", "free_pt_session", "email")
except drafts.OfferUnavailable as e:
    print("Refused:", e)

Refused: Never contact a member more than once in 30 days; The discounted month is for month-to-month members only
Refused: This offer is not available by email


## 3. LLM draft (one call)

Same evidence, rewritten by the model. If the call fails or the output is rejected the cell prints the template (`source: template`).
The endpoint and key are never printed.

In [4]:
# M0155: a discount by email, with the lost-instructor fact in the evidence.
show("M0155", "discount_1m", "email", use_llm=True)

--- M0155 / discount_1m / email [llm]
Subject: A thank you for you
Hi there,

Thank you for joining us recently. We've loved seeing you in our group classes, and we know it's been a change to hear that one of your regular instructors has left the gym. We're grateful for your energy in every session.

As a thank you, we're offering month-to-month members one month at 20% off. Just let us know if you'd like to take advantage of it.

We hope to see you in class soon.

Warm regards,
The Team
Evidence:
  - Offer: One month at 20% off
  - Who it is for: Month-to-month members only
  - Channel: email
  - The member joined very recently
  - The member attends group classes
  - An instructor the member often trains with has left the gym

